# 🌊 Streaming Responses

**Implement real-time streaming for LLM APIs**

---

## 📋 Overview

**What you'll learn:**
- Server-Sent Events (SSE)
- Streaming with OpenAI
- FastAPI streaming endpoints
- Error handling in streams
- Client-side integration

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
# Installation
# !pip install fastapi uvicorn openai sse-starlette

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
import json
import asyncio
import os

client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Stream?

### Non-Streaming vs Streaming:

**Non-Streaming (Bad UX):**
```
User: "Write a long essay"
⏳ Loading... (15 seconds)
💬 [Full response appears at once]

❌ Problems:
- User waits 15 seconds
- No progress feedback
- Feels slow
- Timeout issues
```

**Streaming (Good UX):**
```
User: "Write a long essay"
💬 Once upon a...
💬 Once upon a time...
💬 Once upon a time there was...

✅ Benefits:
- Instant feedback
- Perceived performance
- Better UX
- Can stop early
```

### Use Cases:

- **Chat interfaces** - ChatGPT-style streaming
- **Long-form content** - Articles, essays, code
- **Real-time progress** - Show thinking process
- **Early stopping** - User can stop generation

## 🔄 OpenAI Streaming Basics

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))

async def stream_example():
    """Basic streaming example."""
    
    print("\n🌊 Streaming response:\n")
    
    stream = await async_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": "Count from 1 to 5 slowly."}
        ],
        stream=True  # Enable streaming!
    )
    
    async for chunk in stream:
        # Each chunk contains a delta
        content = chunk.choices[0].delta.content
        
        if content:
            print(content, end='', flush=True)
    
    print("\n\n✅ Stream complete")

# Run example
await stream_example()

## 🎯 Understanding Stream Chunks

In [ ]:
async def inspect_chunks():
    """Inspect stream chunk structure."""
    
    print("\n🔍 Inspecting stream chunks:\n")
    
    stream = await async_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": "Say 'Hello World'"}],
        stream=True
    )
    
    chunk_count = 0
    
    async for chunk in stream:
        chunk_count += 1
        
        print(f"\nChunk {chunk_count}:")
        print(f"  ID: {chunk.id}")
        print(f"  Model: {chunk.model}")
        print(f"  Content: {chunk.choices[0].delta.content}")
        print(f"  Finish reason: {chunk.choices[0].finish_reason}")
        
        if chunk_count >= 3:  # Show first 3 chunks
            print("\n... (more chunks)")
            break

await inspect_chunks()

## 🚀 FastAPI Streaming Endpoint

In [ ]:
print("""
# Save as main.py and run: uvicorn main:app --reload

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from openai import AsyncOpenAI
import json

app = FastAPI(title="Streaming LLM API")
client = AsyncOpenAI()

class ChatRequest(BaseModel):
    message: str
    model: str = "gpt-3.5-turbo"

async def generate_stream(request: ChatRequest):
    \"\"\"Generate streaming response.\"\"\" 
    
    try:
        stream = await client.chat.completions.create(
            model=request.model,
            messages=[{"role": "user", "content": request.message}],
            stream=True
        )
        
        async for chunk in stream:
            content = chunk.choices[0].delta.content
            
            if content:
                # Send as Server-Sent Events
                yield f"data: {json.dumps({'content': content})}\\n\\n"
        
        # Send done signal
        yield f"data: {json.dumps({'done': True})}\\n\\n"
    
    except Exception as e:
        yield f"data: {json.dumps({'error': str(e)})}\\n\\n"

@app.post("/api/stream")
async def stream_chat(request: ChatRequest):
    \"\"\"Stream chat responses.\"\"\" 
    
    return StreamingResponse(
        generate_stream(request),
        media_type="text/event-stream"
    )

# Test with:
# curl -X POST http://localhost:8000/api/stream \\
#   -H "Content-Type: application/json" \\
#   -d '{"message": "Tell me a joke"}'
""")

## 📦 Server-Sent Events (SSE)

In [ ]:
from typing import AsyncGenerator
import json

async def sse_generator(
    messages: list,
    model: str = "gpt-3.5-turbo"
) -> AsyncGenerator[str, None]:
    """Generate SSE-formatted stream."""
    
    try:
        stream = await async_client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        
        async for chunk in stream:
            delta = chunk.choices[0].delta
            
            # Stream content
            if delta.content:
                data = {
                    'type': 'content',
                    'content': delta.content
                }
                yield f"data: {json.dumps(data)}\n\n"
            
            # Stream tool calls (if any)
            if delta.tool_calls:
                data = {
                    'type': 'tool_call',
                    'tool_calls': [tc.model_dump() for tc in delta.tool_calls]
                }
                yield f"data: {json.dumps(data)}\n\n"
            
            # Check if done
            if chunk.choices[0].finish_reason:
                data = {
                    'type': 'done',
                    'finish_reason': chunk.choices[0].finish_reason
                }
                yield f"data: {json.dumps(data)}\n\n"
    
    except Exception as e:
        error_data = {
            'type': 'error',
            'error': str(e)
        }
        yield f"data: {json.dumps(error_data)}\n\n"

print("📦 SSE Generator")
print("\nSSE Format:")
print('data: {"type": "content", "content": "Hello"}\\n\\n')
print('data: {"type": "done", "finish_reason": "stop"}\\n\\n')

## ✅ Complete Streaming API

In [ ]:
print("""
# Complete production-ready streaming API

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, AsyncGenerator
from openai import AsyncOpenAI
import json
import time

app = FastAPI(title="Production Streaming API")

# CORS for browser access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

client = AsyncOpenAI()

class Message(BaseModel):
    role: str
    content: str

class StreamRequest(BaseModel):
    messages: List[Message]
    model: str = Field(default="gpt-3.5-turbo")
    temperature: float = Field(default=0.7, ge=0, le=2)
    max_tokens: int = Field(default=1000, gt=0, le=4000)

async def stream_with_metadata(
    request: StreamRequest
) -> AsyncGenerator[str, None]:
    \"\"\"Stream with metadata tracking.\"\"\" 
    
    start_time = time.time()
    token_count = 0
    
    try:
        # Send start event
        yield f"data: {json.dumps({'type': 'start'})}\\n\\n"
        
        # Create stream
        stream = await client.chat.completions.create(
            model=request.model,
            messages=[m.model_dump() for m in request.messages],
            temperature=request.temperature,
            max_tokens=request.max_tokens,
            stream=True
        )
        
        # Stream chunks
        async for chunk in stream:
            delta = chunk.choices[0].delta
            
            if delta.content:
                token_count += 1
                
                data = {
                    'type': 'content',
                    'content': delta.content,
                    'tokens': token_count
                }
                yield f"data: {json.dumps(data)}\\n\\n"
            
            # Check finish
            if chunk.choices[0].finish_reason:
                latency = time.time() - start_time
                
                data = {
                    'type': 'done',
                    'finish_reason': chunk.choices[0].finish_reason,
                    'tokens': token_count,
                    'latency_ms': round(latency * 1000, 2)
                }
                yield f"data: {json.dumps(data)}\\n\\n"
    
    except Exception as e:
        error_data = {
            'type': 'error',
            'error': str(e),
            'tokens': token_count
        }
        yield f"data: {json.dumps(error_data)}\\n\\n"

@app.post("/api/stream")
async def stream_endpoint(request: StreamRequest):
    \"\"\"Production streaming endpoint.\"\"\" 
    
    return StreamingResponse(
        stream_with_metadata(request),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no"  # Disable nginx buffering
        }
    )

@app.get("/health")
async def health():
    return {"status": "healthy"}
""")

## 🖥️ Client-Side Integration

In [ ]:
print("""
// JavaScript client for streaming API

async function streamChat(message) {
    const response = await fetch('http://localhost:8000/api/stream', {
        method: 'POST',
        headers: {
            'Content-Type': 'application/json',
        },
        body: JSON.stringify({
            messages: [{ role: 'user', content: message }]
        })
    });
    
    const reader = response.body.getReader();
    const decoder = new TextDecoder();
    
    while (true) {
        const { done, value } = await reader.read();
        
        if (done) break;
        
        // Decode chunk
        const chunk = decoder.decode(value);
        
        // Parse SSE data
        const lines = chunk.split('\\n');
        
        for (const line of lines) {
            if (line.startsWith('data: ')) {
                const data = JSON.parse(line.slice(6));
                
                if (data.type === 'content') {
                    // Append to UI
                    document.getElementById('response').innerText += data.content;
                } else if (data.type === 'done') {
                    console.log('Stream done:', data);
                } else if (data.type === 'error') {
                    console.error('Stream error:', data.error);
                }
            }
        }
    }
}

// Usage
streamChat('Tell me a story');
""")

print("\n\n")

print("""
# Python client for streaming API

import requests
import json

def stream_chat(message: str):
    \"\"\"Stream chat with Python client.\"\"\" 
    
    response = requests.post(
        'http://localhost:8000/api/stream',
        json={
            'messages': [{'role': 'user', 'content': message}]
        },
        stream=True  # Important!
    )
    
    for line in response.iter_lines():
        if line:
            line = line.decode('utf-8')
            
            if line.startswith('data: '):
                data = json.loads(line[6:])
                
                if data['type'] == 'content':
                    print(data['content'], end='', flush=True)
                elif data['type'] == 'done':
                    print(f"\\n\\nDone! Tokens: {data['tokens']}")
                elif data['type'] == 'error':
                    print(f"\\nError: {data['error']}")

# Usage
stream_chat('Tell me a joke')
""")

## 🛡️ Error Handling in Streams

In [ ]:
print("""
async def safe_stream(request: StreamRequest):
    \"\"\"Stream with comprehensive error handling.\"\"\" 
    
    try:
        # Validate request
        if not request.messages:
            yield f"data: {json.dumps({'type': 'error', 'error': 'No messages'})}\\n\\n"
            return
        
        # Create stream with timeout
        stream = await asyncio.wait_for(
            client.chat.completions.create(
                model=request.model,
                messages=[m.model_dump() for m in request.messages],
                stream=True
            ),
            timeout=30.0  # 30 second timeout
        )
        
        # Stream with error handling
        async for chunk in stream:
            try:
                delta = chunk.choices[0].delta
                
                if delta.content:
                    yield f"data: {json.dumps({'type': 'content', 'content': delta.content})}\\n\\n"
                
                if chunk.choices[0].finish_reason:
                    yield f"data: {json.dumps({'type': 'done'})}\\n\\n"
            
            except Exception as e:
                # Continue stream even if one chunk fails
                yield f"data: {json.dumps({'type': 'warning', 'message': str(e)})}\\n\\n"
    
    except asyncio.TimeoutError:
        yield f"data: {json.dumps({'type': 'error', 'error': 'Request timeout'})}\\n\\n"
    
    except openai.RateLimitError:
        yield f"data: {json.dumps({'type': 'error', 'error': 'Rate limit exceeded'})}\\n\\n"
    
    except openai.APIError as e:
        yield f"data: {json.dumps({'type': 'error', 'error': f'API error: {str(e)}'})}\\n\\n"
    
    except Exception as e:
        yield f"data: {json.dumps({'type': 'error', 'error': 'Internal error'})}\\n\\n"
    
    finally:
        # Always send final event
        yield f"data: {json.dumps({'type': 'end'})}\\n\\n"

💡 Key Error Handling:
  - Validate input before streaming
  - Set timeouts
  - Handle per-chunk errors
  - Always send final event
  - Distinguish error types
""")

## ✅ Summary

### Streaming Basics:

**1. Enable Streaming**
```python
stream = await client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[...],
    stream=True  # ← This!
)
```

**2. Process Chunks**
```python
async for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        yield content
```

**3. FastAPI Endpoint**
```python
@app.post("/stream")
async def stream(request):
    return StreamingResponse(
        generate_stream(request),
        media_type="text/event-stream"
    )
```

### SSE Format:

```
data: {"type": "content", "content": "Hello"}\n\n
data: {"type": "content", "content": " World"}\n\n
data: {"type": "done", "finish_reason": "stop"}\n\n
```

**Rules:**
- Prefix with `data: `
- End with `\n\n`
- Use JSON for structure
- Send types: content, done, error

### Best Practices:

**1. Add Metadata**
```python
yield f"data: {json.dumps({
    'type': 'content',
    'content': content,
    'tokens': token_count,
    'timestamp': time.time()
})}\n\n"
```

**2. Handle Errors**
```python
try:
    async for chunk in stream:
        yield chunk
except Exception as e:
    yield f"data: {json.dumps({'type': 'error', 'error': str(e)})}\\n\\n"
```

**3. Set Headers**
```python
headers={
    "Cache-Control": "no-cache",
    "X-Accel-Buffering": "no"  # Nginx
}
```

**4. Add CORS**
```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"]
)
```

### Common Issues:

**Issue: Buffering**
```python
# Solution: Disable buffering
headers={"X-Accel-Buffering": "no"}
```

**Issue: CORS Errors**
```python
# Solution: Add CORS middleware
app.add_middleware(CORSMiddleware, ...)
```

**Issue: Connection Drops**
```python
# Solution: Handle in try/finally
try:
    async for chunk in stream:
        yield chunk
finally:
    # Cleanup
```

### Testing:

**cURL:**
```bash
curl -X POST http://localhost:8000/api/stream \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Hi"}]}'
```

**Python:**
```python
import requests

response = requests.post(
    'http://localhost:8000/api/stream',
    json={'messages': [...]},
    stream=True
)

for line in response.iter_lines():
    print(line.decode())
```

### Next: `08_production_apis/03_error_handling.ipynb`